# College Track: All Performance Spaghetti Plots, 2016–2026

Each line follows one athlete. **Every valid numeric result is plotted**, rather than only an annual best. Sprint and hurdle results use seconds; long jump uses meters.

**X-axis:** Year 1 is the athlete’s first recorded year across these four event files. Year 2 is the following calendar year, and so on. This is an observed school-year timeline, not verified freshman/sophomore eligibility. An athlete already competing in 2016 may have started earlier. Missing years break lines.

Results within the same school year share the same x-position, so multiple meets form vertical segments. Points are ordered by meet date when available; tied dates retain file order. Name matching handles “Last, First” and “First Last”, but identical names may refer to different people.

## Start here
1. Save this notebook in your repository root or its `notebooks` folder.
2. Keep your four CSVs in `data/processed/by_event`.
3. Choose **Kernel → Restart & Run All**. If data cannot be found, enter the exact folder path in `DATA_DIR` below.
4. Save after running so GitHub can display the plot outputs.

Dependencies: pandas, numpy, matplotlib, and Jupyter. If needed, run `%pip install pandas numpy matplotlib` in a new cell.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# Optional: paste the folder containing all four CSVs here.
# Example: Path.home() / "Projects/collegetrackperformances/data/processed/by_event"
DATA_DIR = None
FILES = {"100H": "100H_2016_2026.csv", "400H": "400H_2016_2026.csv",
         "400m": "400m_2016_2026.csv", "Long Jump": "long_jump_2016_2026.csv"}
# Display range only: results outside it remain in the data and audit.
Y_LIMITS = {"100H": (12, 17), "400H": None, "400m": None, "Long Jump": None}
SAVE_PLOTS = True

if DATA_DIR is not None:
    DATA_DIR = Path(DATA_DIR).expanduser()
else:
    candidates = []
    for p in [Path.cwd(), *Path.cwd().parents]:
        candidates += [p / "data/processed/by_event", p]
    DATA_DIR = next((p for p in candidates if all((p / f).is_file() for f in FILES.values())), None)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing all four event CSVs, then run again.")
missing = [f for f in FILES.values() if not (DATA_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f"Missing CSVs in {DATA_DIR}: {missing}")
OUTPUT_DIR = Path.cwd() / "spaghetti_outputs"
if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(exist_ok=True)
print("Data folder:", DATA_DIR)


## Prepare the results
Numeric times, including `1:02.34`, are converted to seconds. Long jump supports meter values and feet/inch values when explicitly labeled. DNF, DNS, fouls, missing names, unknown jump units, and other unparseable results are excluded and counted below. Exact repeated records are retained because they may represent separate rounds or attempts.

Wind-assisted and unknown-wind results are included. Conference and school labels are used as recorded in the files; these may not reflect historical conference membership.

In [ ]:
def athlete_name(value):
    if pd.isna(value) or not str(value).strip():
        return None
    value = str(value).strip()
    if "," in value:
        last, first = value.split(",", 1)
        value = f"{first.strip()} {last.strip()}"
    return " ".join(value.split()).casefold()

def numeric_result(value, event, unit):
    s = str(value).strip()
    s = re.sub(r"[aAwW]$", "", s).strip()
    try:
        if event != "Long Jump":
            parts = s.split(":")
            return sum(float(v) * 60**i for i, v in enumerate(reversed(parts))) if len(parts) <= 3 else np.nan
        u = str(unit).strip().lower()
        if u in {"m", "meter", "meters", "metre", "metres"}:
            return float(s)
        if u in {"ft", "feet", "foot", "ft-in", "feet/inches", "feet and inches"}:
            match = re.fullmatch(r"(\d+)['’\-]\s*(\d+(?:\.\d+)?)[\"”]?", s)
            return (float(match[1])*12 + float(match[2]))*0.0254 if match else float(s)*0.3048
        return np.nan
    except (ValueError, TypeError):
        return np.nan

frames, audits = [], []
for event, filename in FILES.items():
    d = pd.read_csv(DATA_DIR / filename)
    d.columns = d.columns.str.strip().str.lower()
    required = {"athlete", "season", "performance"}
    if not required.issubset(d.columns):
        raise ValueError(f"{filename}: missing columns {required - set(d.columns)}")
    for col in ["school", "conference", "unit", "meet_date"]:
        if col not in d:
            d[col] = np.nan
    d["event"] = event
    d["season"] = pd.to_numeric(d["season"], errors="coerce")
    d["athlete_key"] = d["athlete"].map(athlete_name)
    d["value"] = [numeric_result(v, event, u) for v, u in zip(d.performance, d.unit)]
    d["date"] = pd.to_datetime(d["meet_date"], errors="coerce")
    d["record_order"] = np.arange(len(d))
    valid = (d.season.between(2016, 2026) & d.season.mod(1).eq(0)
             & d.athlete_key.notna() & np.isfinite(d.value) & d.value.gt(0))
    audits.append({"Event": event, "Input records": len(d), "Plotted records": int(valid.sum()),
                   "Excluded records": int((~valid).sum())})
    frames.append(d.loc[valid].copy())
results = pd.concat(frames, ignore_index=True)
if results.empty:
    raise ValueError("No valid results. Check CSV columns and units.")
results["first_year"] = results.groupby("athlete_key")["season"].transform("min")
results["school_year"] = (results.season - results.first_year + 1).astype(int)
for col in ["school", "conference"]:
    results[col] = results[col].fillna("Unknown").astype(str).str.strip().replace("", "Unknown")
audit = pd.DataFrame(audits)
display(audit)
display(results.groupby("event").agg(athletes=("athlete_key", "nunique"),
                                    records=("value", "size"),
                                    first_calendar_year=("season", "min"),
                                    last_calendar_year=("season", "max")))


## All athletes, all results
Each event has its own plot. Lower times indicate better sprint/hurdle performances; greater distances indicate better jumps. Colors group athletes by their first recorded conference in that event, solely to help readability. Transfer athletes keep the same line. Single-result athletes appear as dots.

100H displays 12–17 seconds, as requested. The printed count identifies records outside that visible range. Set `Y_LIMITS["100H"] = None` to display its full range.

In [ ]:
def spaghetti(event, subset=None, title=None, legend=True):
    d = results[results.event.eq(event)].copy() if subset is None else subset.copy()
    fig, ax = plt.subplots(figsize=(16, 9))
    if d.empty:
        ax.text(.5, .5, "No valid results", transform=ax.transAxes, ha="center")
    conferences = sorted(d.conference.unique())
    colors = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(conferences)}
    for _, g in d.groupby("athlete_key", sort=True):
        g = g.sort_values(["season", "date", "record_order"], na_position="last", kind="stable")
        color = colors[g.iloc[0].conference]
        # Separate runs across missing years; keep all results within each run.
        blocks = g.season.diff().gt(1).cumsum()
        for _, run in g.groupby(blocks):
            ax.plot(run.school_year, run.value, color=color, alpha=.38,
                    linewidth=1.15, marker="o", markersize=3)
    max_year = int(results.school_year.max())
    ax.set_xticks(range(1, max_year + 1), [f"Year {i}" for i in range(1, max_year + 1)])
    ax.set_xlim(.7, max_year + .3)
    ax.set_xlabel("School year since first recorded appearance", fontsize=13)
    ax.set_ylabel("Distance (meters) — higher is better" if event == "Long Jump"
                  else "Time (seconds) — lower is better", fontsize=13)
    if Y_LIMITS[event] is not None:
        ax.set_ylim(*Y_LIMITS[event])
    ax.set_title(title or f"{event}: All Athlete Performances", fontsize=21, weight="bold", pad=18)
    ax.grid(alpha=.18)
    ax.spines[["top", "right"]].set_visible(False)
    if legend and conferences:
        ax.legend(handles=[Line2D([0], [0], color=colors[k], lw=3, label=k) for k in conferences],
                  title="First recorded conference in this event", loc="upper right")
    fig.text(.08, .02, f"{d.athlete_key.nunique():,} athletes | {len(d):,} results | "
             "Every valid result retained; vertical segments show results within a year.", fontsize=10)
    fig.tight_layout(rect=[0, .045, 1, 1])
    return fig

for event in FILES:
    d = results[results.event.eq(event)]
    if Y_LIMITS[event]:
        low, high = Y_LIMITS[event]
        print(event, "results outside the displayed range:", int((~d.value.between(low, high)).sum()))
    fig = spaghetti(event)
    if SAVE_PLOTS:
        fig.savefig(OUTPUT_DIR / f"{event.replace(' ', '_')}_all_results.png", dpi=180)
    plt.show()
    plt.close(fig)


## Compare schools and conferences
These additional plots include every school and conference present in the files. Each figure uses the same school-year alignment. School plots show results earned at that school; an athlete who transfers may appear in more than one figure. Their Year 1 baseline stays unchanged. Conference plots use each result’s recorded conference.

In [ ]:
# Set False if you only want the four overall plots.
MAKE_GROUP_PLOTS = True
if MAKE_GROUP_PLOTS:
    for group_col in ["conference", "school"]:
        for event in FILES:
            event_data = results[results.event.eq(event)]
            for group, subset in event_data.groupby(group_col, sort=True):
                fig = spaghetti(event, subset, f"{event} — {group}: All Results", legend=False)
                if SAVE_PLOTS:
                    safe = re.sub(r"[^A-Za-z0-9_-]+", "_", str(group))
                    fig.savefig(OUTPUT_DIR / f"{group_col}_{safe}_{event.replace(' ', '_')}.png", dpi=150)
                plt.show()
                plt.close(fig)


## Observations and coverage
The table below counts each athlete’s results and observed school years. The change compares average results in the first and last observed year; it is **not** a comparison of every intermediate year. Negative time changes indicate improvement; positive long-jump changes indicate improvement. One-year athletes have no longitudinal change.

Use the plots to describe whether performances improve consistently or vary from meet to meet. Avoid interpreting missing years as poor performances. Wind, meet conditions, number of attempts/races, and incomplete coverage can affect comparisons.

In [ ]:
year_means = results.groupby(["event", "athlete_key", "school_year"]).value.mean().reset_index()
rows = []
for (event, athlete), g in year_means.groupby(["event", "athlete_key"]):
    g = g.sort_values("school_year")
    raw = results[results.event.eq(event) & results.athlete_key.eq(athlete)]
    change = g.iloc[-1].value - g.iloc[0].value if len(g) > 1 else np.nan
    rows.append({"event": event, "athlete": athlete, "records": len(raw), "observed_years": len(g),
                 "first_school_year": int(g.iloc[0].school_year), "last_school_year": int(g.iloc[-1].school_year),
                 "first_year_mean": g.iloc[0].value, "last_year_mean": g.iloc[-1].value,
                 "change": change,
                 "direction": "One observed year" if pd.isna(change) else
                 "Improved" if (change > 0 if event == "Long Jump" else change < 0) else
                 "Unchanged" if change == 0 else "Declined"})
comparison = pd.DataFrame(rows)
display(comparison.round(3))
display(pd.crosstab(comparison.event, comparison.direction))
if SAVE_PLOTS:
    comparison.to_csv(OUTPUT_DIR / "all_athlete_comparisons.csv", index=False)
    audit.to_csv(OUTPUT_DIR / "record_audit.csv", index=False)


## Put the notebook on GitHub
After **Run All**, save the notebook with **Command + S**. Move it into your repository’s `notebooks` folder, then run these commands in Terminal from the repository root:

```bash
mkdir -p notebooks
cp ~/Downloads/college_track_all_results_spaghetti.ipynb notebooks/
git add notebooks/college_track_all_results_spaghetti.ipynb
git commit -m "Add all-result athlete spaghetti plots for four track events"
git push
```

If you already saved the notebook in `notebooks`, skip the `cp` command. To publish exported figures and comparison tables too, move `spaghetti_outputs` into the repository if needed and run `git add spaghetti_outputs/` before committing. The CSV data must remain available at the expected relative path for other people to rerun the notebook.

This notebook ships without executed data outputs; run it with your actual CSVs before committing to show the charts on GitHub.